In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CELL 1 ── Install libraries
# ══════════════════════════════════════════════════════════════════════

!pip install -q imagehash Pillow tqdm scikit-learn


# ══════════════════════════════════════════════════════════════════════
# CELL 3 ── Imports & Configuration  ← only edit this cell
# ══════════════════════════════════════════════════════════════════════

import os, re, shutil, hashlib
from pathlib import Path

import imagehash
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split

# ── Paths ─────────────────────────────────────────────────────────────
# Folder that contains one sub-folder per SeriesInstanceUID → .jpg files
JPEG_BASE  = Path("/content/drive/MyDrive/Breast_cancer_dataset/jpeg")

# Folder where your 5 CSV files live
CSV_DIR    = Path("/content/drive/MyDrive/Breast_cancer_dataset/csv")

# Where the split output will be created
OUTPUT_DIR = Path("/content/drive/MyDrive/Breast_cancer_dataset/cropped_split")

# ── Split ratios ──────────────────────────────────────────────────────
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15
RANDOM_SEED = 42

# ── Duplicate detection ───────────────────────────────────────────────
# Perceptual-hash Hamming distance ≤ this value → treat as near-duplicate
PHASH_THRESHOLD = 8

# ── Label mapping ─────────────────────────────────────────────────────
# BENIGN_WITHOUT_CALLBACK → benign  (standard practice)
LABEL_MAP = {
    "MALIGNANT"              : 1,
    "BENIGN"                 : 0,
    "BENIGN_WITHOUT_CALLBACK": 0,
}

# ─────────────────────────────────────────────────────────────────────
assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-6, \
    "Ratios must sum to 1.0"

print("=" * 65)
print("  CBIS-DDSM  —  Cropped Images Split Pipeline")
print("=" * 65)
print(f"  JPEG root  : {JPEG_BASE}")
print(f"  CSV dir    : {CSV_DIR}")
print(f"  Output dir : {OUTPUT_DIR}")
print(f"  Split      : train={TRAIN_RATIO:.0%}  val={VAL_RATIO:.0%}  test={TEST_RATIO:.0%}")
print(f"  Seed       : {RANDOM_SEED}")
print()


# ══════════════════════════════════════════════════════════════════════
# CELL 4 ── Load CSVs & build master metadata for cropped images
# ══════════════════════════════════════════════════════════════════════

print("▶  STEP 1 — Loading CSVs …")

# ── Load ──────────────────────────────────────────────────────────────
dicom_info = pd.read_csv(CSV_DIR / "dicom_info.csv")
calc_train = pd.read_csv(CSV_DIR / "calc_case_description_train_set.csv")
calc_test  = pd.read_csv(CSV_DIR / "calc_case_description_test_set.csv")
mass_train = pd.read_csv(CSV_DIR / "mass_case_description_train_set.csv")
mass_test  = pd.read_csv(CSV_DIR / "mass_case_description_test_set.csv")

# ── Normalise column names ────────────────────────────────────────────
def norm_cols(df):
    df = df.copy()
    df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
    return df

calc_train = norm_cols(calc_train)
calc_test  = norm_cols(calc_test)
mass_train = norm_cols(mass_train)
mass_test  = norm_cols(mass_test)

calc_train["lesion_type"] = "calcification";  calc_train["orig_split"] = "train"
calc_test["lesion_type"]  = "calcification";  calc_test["orig_split"]  = "test"
mass_train["lesion_type"] = "mass";           mass_train["orig_split"] = "train"
mass_test["lesion_type"]  = "mass";           mass_test["orig_split"]  = "test"

# ── Combine all case rows ─────────────────────────────────────────────
case_all = pd.concat(
    [calc_train, calc_test, mass_train, mass_test],
    ignore_index=True
)

# The image_file_path column looks like:
#   "Mass-Training_P_00001_LEFT_CC/UID/UID/000000.dcm"
# The first segment is the "case folder" — the join key.
case_all["case_folder"] = case_all["image_file_path"].str.split("/").str[0]

# One metadata row per case folder (dedup keeps first occurrence)
case_meta = (
    case_all[[
        "case_folder", "patient_id", "pathology",
        "lesion_type", "orig_split", "assessment", "subtlety"
    ]]
    .drop_duplicates(subset="case_folder")
    .reset_index(drop=True)
)

print(f"  Case rows loaded           : {len(case_all):,}")
print(f"  Unique case folders        : {case_meta['case_folder'].nunique():,}")
print(f"  Unique patients            : {case_meta['patient_id'].nunique():,}")

# ── Filter dicom_info for CROPPED images only ─────────────────────────
#
# dicom_info.PatientID looks like:
#   "Mass-Training_P_00001_LEFT_CC_1"  (may have _1, _2 abnormality suffix)
# Strip that trailing _<digit> to get the case folder.
#
dicom_info["case_folder"] = dicom_info["PatientID"].astype(str).apply(
    lambda x: re.sub(r"_\d+$", "", x)
)

cropped_dicom = dicom_info[
    dicom_info["SeriesDescription"] == "cropped images"
].copy().reset_index(drop=True)

print(f"\n  Total images in dicom_info : {len(dicom_info):,}")
print(f"  Cropped images found       : {len(cropped_dicom):,}")

# ── Build Drive file paths ─────────────────────────────────────────────
# dicom image_path format: "CBIS-DDSM/jpeg/<SeriesUID>/<file>.jpg"
# Drive path             : JPEG_BASE / <SeriesUID> / <file>.jpg
cropped_dicom["rel_path"]   = cropped_dicom["image_path"].str.replace(
    r"^CBIS-DDSM/jpeg/", "", regex=True
)
cropped_dicom["drive_path"] = cropped_dicom["rel_path"].apply(
    lambda x: str(JPEG_BASE / x)
)
cropped_dicom["filename"]   = cropped_dicom["image_path"].str.split("/").str[-1]

# ── Merge dicom with case metadata ────────────────────────────────────
master = cropped_dicom.merge(case_meta, on="case_folder", how="left")
master["label"]     = master["pathology"].map(LABEL_MAP)
master["label_str"] = master["pathology"].fillna("unknown")

null_path = master["pathology"].isna().sum()

print(f"\n  After merge shape          : {master.shape}")
print(f"  Images with null pathology : {null_path}")
print(f"\n  Pathology distribution:")
print("    " + master["pathology"].value_counts(dropna=False)
      .to_string().replace("\n", "\n    "))
print(f"\n  Lesion type distribution:")
print("    " + master["lesion_type"].value_counts(dropna=False)
      .to_string().replace("\n", "\n    "))
print(f"\n  Unique patients            : {master['patient_id'].nunique():,}")


# ══════════════════════════════════════════════════════════════════════
# CELL 5 ── Verify files exist on Drive
# ══════════════════════════════════════════════════════════════════════

print("\n▶  STEP 2 — Checking files exist on Drive …")

master["file_exists"] = [
    os.path.isfile(p)
    for p in tqdm(master["drive_path"], desc="  Checking", unit="file")
]

n_found   = master["file_exists"].sum()
n_missing = (~master["file_exists"]).sum()

print(f"  Files found   : {n_found:,}")
print(f"  Files MISSING : {n_missing:,}")

if n_missing > 0:
    print(f"\n  ⚠  Missing file sample (first 5):")
    miss = master[~master["file_exists"]]["drive_path"].head(5).tolist()
    for p in miss:
        print(f"    {p}")
    print("  → These will be SKIPPED. Check your JPEG_BASE path if count is high.")

# Keep only files that exist
master = master[master["file_exists"]].copy().reset_index(drop=True)
print(f"\n  Proceeding with {len(master):,} images")


# ══════════════════════════════════════════════════════════════════════
# CELL 6 ── Exact duplicate removal  (MD5 hash)
# ══════════════════════════════════════════════════════════════════════

print("\n▶  STEP 3 — Exact duplicate removal (MD5) …")

def md5_hash(path: str) -> str:
    h = hashlib.md5()
    try:
        with open(path, "rb") as f:
            for chunk in iter(lambda: f.read(65536), b""):
                h.update(chunk)
        return h.hexdigest()
    except Exception:
        return f"READ_ERROR"

master["md5"] = [
    md5_hash(p)
    for p in tqdm(master["drive_path"], desc="  Computing MD5", unit="file")
]

before          = len(master)
dup_md5_mask    = master.duplicated(subset="md5", keep="first")
n_exact_removed = dup_md5_mask.sum()
master          = master[~dup_md5_mask].copy().reset_index(drop=True)

print(f"  Exact duplicates removed : {n_exact_removed:,}")
print(f"  Remaining images         : {len(master):,}")


# ══════════════════════════════════════════════════════════════════════
# CELL 7 ── Near-duplicate removal  (perceptual hash)
# ══════════════════════════════════════════════════════════════════════

print(f"\n▶  STEP 4 — Near-duplicate removal (pHash ≤ {PHASH_THRESHOLD}) …")
print("  This catches visually identical images that differ in compression or")
print("  minor pixel changes — common in medical imaging datasets.\n")

def compute_phash(path: str):
    try:
        return imagehash.phash(Image.open(path).convert("L"))
    except Exception:
        return None

phashes = [
    compute_phash(p)
    for p in tqdm(master["drive_path"], desc="  Computing pHash", unit="file")
]
master["phash"] = phashes

# Mark near-duplicates: keep first, remove subsequent
keep = [True] * len(master)

for i in tqdm(range(len(master)), desc="  Comparing", unit="img"):
    if not keep[i] or phashes[i] is None:
        continue
    for j in range(i + 1, len(master)):
        if not keep[j] or phashes[j] is None:
            continue
        try:
            if phashes[i] - phashes[j] <= PHASH_THRESHOLD:
                keep[j] = False
        except Exception:
            pass

before           = len(master)
n_near_removed   = keep.count(False)
master           = master[keep].copy().reset_index(drop=True)

print(f"  Near-duplicates removed  : {n_near_removed:,}")
print(f"  Remaining images         : {len(master):,}")
print(f"\n  Final label distribution:")
print("    " + master["pathology"].value_counts(dropna=False)
      .to_string().replace("\n", "\n    "))


# ══════════════════════════════════════════════════════════════════════
# CELL 8 ── Patient-level stratified split  +  leakage audit
# ══════════════════════════════════════════════════════════════════════

print("\n▶  STEP 5 — Patient-level stratified split …")
print()
print("  ┌─ Why patient-level? ──────────────────────────────────────────┐")
print("  │  Each patient can have multiple cropped images (different     │")
print("  │  views, different abnormalities). Splitting by IMAGE would    │")
print("  │  put the same patient in both train and test → the model      │")
print("  │  learns patient anatomy, not tumour features → data leakage. │")
print("  │  Splitting by PATIENT guarantees zero overlap.                │")
print("  └───────────────────────────────────────────────────────────────┘")
print()

# Separate labelled vs unlabelled rows (unlabelled = null pathology)
df_labeled   = master[master["label"].notna()].copy().reset_index(drop=True)
df_unlabeled = master[master["label"].isna()].copy().reset_index(drop=True)

print(f"  Labelled images   : {len(df_labeled):,}")
print(f"  Unlabelled images : {len(df_unlabeled):,}")

# ── One label per patient ────────────────────────────────────────────
# If a patient has ANY malignant image → label the patient as malignant
patient_labels = (
    df_labeled
    .groupby("patient_id")["label"]
    .agg(lambda x: 1 if (x == 1).any() else 0)
    .reset_index()
    .rename(columns={"label": "patient_label"})
)

patients   = patient_labels["patient_id"].values
pat_labels = patient_labels["patient_label"].values

n_patients   = len(patients)
n_malignant  = (pat_labels == 1).sum()
n_benign     = (pat_labels == 0).sum()

print(f"\n  Total unique patients : {n_patients:,}")
print(f"  Malignant patients    : {n_malignant:,}  ({n_malignant/n_patients:.1%})")
print(f"  Benign patients       : {n_benign:,}  ({n_benign/n_patients:.1%})")

# ── Stratified split at patient level ────────────────────────────────
val_test_ratio = VAL_RATIO + TEST_RATIO

pat_train, pat_temp, lbl_train, lbl_temp = train_test_split(
    patients, pat_labels,
    test_size    = val_test_ratio,
    stratify     = pat_labels,
    random_state = RANDOM_SEED,
)

# Split the held-out pool into val + test
val_fraction = VAL_RATIO / val_test_ratio

pat_val, pat_test, lbl_val, lbl_test = train_test_split(
    pat_temp, lbl_temp,
    test_size    = 1 - val_fraction,
    stratify     = lbl_temp,
    random_state = RANDOM_SEED,
)

pat_train_set = set(pat_train)
pat_val_set   = set(pat_val)
pat_test_set  = set(pat_test)

# ── Assign split to each image ────────────────────────────────────────
def assign_split(pid):
    if pid in pat_train_set: return "train"
    if pid in pat_val_set:   return "val"
    if pid in pat_test_set:  return "test"
    return "unassigned"

df_labeled["split"] = df_labeled["patient_id"].apply(assign_split)

# ── DATA LEAKAGE AUDIT ───────────────────────────────────────────────
print()
print("  ━" * 32)
print("  🔒  DATA LEAKAGE AUDIT")
print("  ━" * 32)

train_val_overlap  = pat_train_set & pat_val_set
train_test_overlap = pat_train_set & pat_test_set
val_test_overlap   = pat_val_set   & pat_test_set

img_train = set(df_labeled[df_labeled["split"] == "train"]["drive_path"])
img_val   = set(df_labeled[df_labeled["split"] == "val"]["drive_path"])
img_test  = set(df_labeled[df_labeled["split"] == "test"]["drive_path"])

print()
print("  Patient-level overlap:")
print(f"    Train ∩ Val   = {len(train_val_overlap):>3} patients  (must be 0)")
print(f"    Train ∩ Test  = {len(train_test_overlap):>3} patients  (must be 0)")
print(f"    Val   ∩ Test  = {len(val_test_overlap):>3} patients  (must be 0)")

print()
print("  Image-level overlap:")
print(f"    Train ∩ Val   = {len(img_train & img_val):>3} images  (must be 0)")
print(f"    Train ∩ Test  = {len(img_train & img_test):>3} images  (must be 0)")
print(f"    Val   ∩ Test  = {len(img_val   & img_test):>3} images  (must be 0)")

# Hard assertions — will crash with a clear message if leakage exists
assert len(train_val_overlap)  == 0, "🚨 LEAKAGE DETECTED: train ∩ val patients"
assert len(train_test_overlap) == 0, "🚨 LEAKAGE DETECTED: train ∩ test patients"
assert len(val_test_overlap)   == 0, "🚨 LEAKAGE DETECTED: val ∩ test patients"
assert len(img_train & img_val)  == 0, "🚨 LEAKAGE DETECTED: train ∩ val images"
assert len(img_train & img_test) == 0, "🚨 LEAKAGE DETECTED: train ∩ test images"
assert len(img_val   & img_test) == 0, "🚨 LEAKAGE DETECTED: val ∩ test images"

print()
print("  ✅  ZERO LEAKAGE — no patient or image appears in more than one split!")
print("  ━" * 32)

# ── Per-split summary ────────────────────────────────────────────────
print()
print(f"  {'Split':<7} {'Images':>7} {'Malignant':>10} {'Benign':>8} {'Patients':>10}")
print(f"  {'─'*47}")

for sname in ["train", "val", "test"]:
    sub = df_labeled[df_labeled["split"] == sname]
    mal = (sub["label"] == 1).sum()
    ben = (sub["label"] == 0).sum()
    pts = sub["patient_id"].nunique()
    print(f"  {sname:<7} {len(sub):>7,} {mal:>10,} {ben:>8,} {pts:>10,}")

print()
print(f"  {'TOTAL':<7} {len(df_labeled):>7,} "
      f"{(df_labeled['label']==1).sum():>10,} "
      f"{(df_labeled['label']==0).sum():>8,} "
      f"{df_labeled['patient_id'].nunique():>10,}")

# Class balance per split
print()
print("  Class balance check:")
for sname in ["train", "val", "test"]:
    sub = df_labeled[df_labeled["split"] == sname]
    mal = (sub["label"] == 1).sum()
    ben = (sub["label"] == 0).sum()
    ratio = max(mal, ben) / max(min(mal, ben), 1)
    flag  = "⚠  imbalanced" if ratio > 1.5 else "✅ balanced"
    print(f"    {sname:<6}  malignant={mal:>4}  benign={ben:>4}  "
          f"ratio={ratio:.2f}x  {flag}")


# ══════════════════════════════════════════════════════════════════════
# CELL 9 ── Copy images into split folder structure
# ══════════════════════════════════════════════════════════════════════

print("\n▶  STEP 6 — Copying images to split_dataset/ …")
print(f"  Output: {OUTPUT_DIR}")
print()
print("  Folder layout:")
print("  cropped_split/")
print("  ├── train/  malignant/  benign/")
print("  ├── val/    malignant/  benign/")
print("  └── test/   malignant/  benign/")
print()

LABEL_FOLDER = {1: "malignant", 0: "benign"}

def safe_copy(src: str, dst: Path) -> str:
    """Copy file, resolving name collisions automatically."""
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():                          # handle collision
        stem, suffix, i = dst.stem, dst.suffix, 1
        while dst.exists():
            dst = dst.parent / f"{stem}_{i}{suffix}"
            i  += 1
    shutil.copy2(src, dst)
    return str(dst)

new_paths = []
errors    = []

for _, row in tqdm(df_labeled.iterrows(), total=len(df_labeled),
                   desc="  Copying", unit="img"):
    label_folder = LABEL_FOLDER.get(int(row["label"]), "unknown")
    dst          = OUTPUT_DIR / row["split"] / label_folder / row["filename"]
    try:
        new_paths.append(safe_copy(row["drive_path"], dst))
    except Exception as e:
        new_paths.append(f"FAILED: {e}")
        errors.append(str(row["drive_path"]))

df_labeled["new_filepath"] = new_paths

print(f"\n  Copied : {len(new_paths) - len(errors):,}")
print(f"  Errors : {len(errors):,}")
if errors:
    print("  Error sample:")
    for e in errors[:3]:
        print(f"    {e}")


# ══════════════════════════════════════════════════════════════════════
# CELL 10 ── Save CSV manifests
# ══════════════════════════════════════════════════════════════════════

print("\n▶  STEP 7 — Saving CSV manifests …")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_COLS = [
    "new_filepath", "filename", "patient_id",
    "label", "label_str", "pathology",
    "lesion_type", "orig_split", "split",
    "assessment", "subtlety", "md5", "drive_path",
]

for sname in ["train", "val", "test"]:
    sub     = df_labeled[df_labeled["split"] == sname][MANIFEST_COLS]
    out_csv = OUTPUT_DIR / f"{sname}_manifest.csv"
    sub.to_csv(out_csv, index=False)
    print(f"  ✔ {sname}_manifest.csv  ({len(sub):,} rows)")

full_csv = OUTPUT_DIR / "full_manifest.csv"
df_labeled[MANIFEST_COLS].to_csv(full_csv, index=False)
print(f"  ✔ full_manifest.csv  ({len(df_labeled):,} rows)")


# ══════════════════════════════════════════════════════════════════════
# CELL 11 ── Final audit report
# ══════════════════════════════════════════════════════════════════════

print()
print("=" * 65)
print("  FINAL AUDIT REPORT")
print("=" * 65)

n_start     = 3567     # total cropped images in dicom_info
n_found     = master["file_exists"].sum() if "file_exists" in master else len(master)

print(f"""
  ┌─ Image pipeline ──────────────────────────────────────┐
  │  Cropped images in dicom_info     : {n_start:>5,}           │
  │  Files found on Drive             : {len(master) + n_exact_removed + n_near_removed:>5,}           │
  │  After exact-duplicate removal    : {len(master) + n_near_removed:>5,}  (-{n_exact_removed})     │
  │  After near-duplicate removal     : {len(master):>5,}  (-{n_near_removed})     │
  │  Labelled images (used)           : {len(df_labeled):>5,}           │
  └───────────────────────────────────────────────────────┘
""")

print(f"  {'Split':<7} {'Images':>7} {'Malignant':>10} {'Benign':>8} {'Patients':>10}  Mal%")
print(f"  {'─'*55}")
for sname in ["train", "val", "test"]:
    sub  = df_labeled[df_labeled["split"] == sname]
    mal  = (sub["label"] == 1).sum()
    ben  = (sub["label"] == 0).sum()
    pts  = sub["patient_id"].nunique()
    malp = mal / len(sub) * 100
    print(f"  {sname:<7} {len(sub):>7,} {mal:>10,} {ben:>8,} {pts:>10,}  {malp:.1f}%")

print()
print("  🔒 Leakage audit:")
print(f"    Patient-level  Train∩Val={len(pat_train_set&pat_val_set)}  "
      f"Train∩Test={len(pat_train_set&pat_test_set)}  "
      f"Val∩Test={len(pat_val_set&pat_test_set)}")
print(f"    Image-level    Train∩Val={len(img_train&img_val)}  "
      f"Train∩Test={len(img_train&img_test)}  "
      f"Val∩Test={len(img_val&img_test)}")
print(f"    ✅ Zero leakage confirmed")

print(f"""
  Output: {OUTPUT_DIR}
  cropped_split/
  ├── train/
  │   ├── malignant/  ({(df_labeled[df_labeled['split']=='train']['label']==1).sum():,} images)
  │   └── benign/     ({(df_labeled[df_labeled['split']=='train']['label']==0).sum():,} images)
  ├── val/
  │   ├── malignant/  ({(df_labeled[df_labeled['split']=='val']['label']==1).sum():,} images)
  │   └── benign/     ({(df_labeled[df_labeled['split']=='val']['label']==0).sum():,} images)
  └── test/
      ├── malignant/  ({(df_labeled[df_labeled['split']=='test']['label']==1).sum():,} images)
      └── benign/     ({(df_labeled[df_labeled['split']=='test']['label']==0).sum():,} images)

  Manifests saved:
    train_manifest.csv | val_manifest.csv | test_manifest.csv | full_manifest.csv
""")

print("=" * 65)
print("  ✅  Split complete! Ready for preprocessing & model training.")
print("=" * 65)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 4.6 MB/s eta 0:00:00
  CBIS-DDSM  —  Cropped Images Split Pipeline
  JPEG root  : /content/drive/MyDrive/Breast_cancer_dataset/jpeg
  CSV dir    : /content/drive/MyDrive/Breast_cancer_dataset/csv
  Output dir : /content/drive/MyDrive/Breast_cancer_dataset/cropped_split
  Split      : train=70%  val=15%  test=15%
  Seed       : 42

▶  STEP 1 — Loading CSVs …
  Case rows loaded           : 3,568
  Unique case folders        : 3,103
  Unique patients            : 1,566

  Total images in dicom_info : 10,237
  Cropped images found       : 3,567

  After merge shape          : (3567, 50)
  Images with null pathology : 0

  Pathology distribution:
    pathology
    MALIGNANT                  1450
    BENIGN                     1438
    BENIGN_WITHOUT_CALLBACK     679

  Lesion type distribution:
    lesion_type
    calcification    1871
    mass             1696

  Unique patients            : 1,566

▶  STEP 2 — Checking files exist 

  Checking: 100%|██████████| 3567/3567 [13:25<00:00,  4.43file/s]


  Files found   : 3,567
  Files MISSING : 0

  Proceeding with 3,567 images

▶  STEP 3 — Exact duplicate removal (MD5) …


  Computing MD5: 100%|██████████| 3567/3567 [17:01<00:00,  3.49file/s]


  Exact duplicates removed : 0
  Remaining images         : 3,567

▶  STEP 4 — Near-duplicate removal (pHash ≤ 8) …
  This catches visually identical images that differ in compression or
  minor pixel changes — common in medical imaging datasets.



  Comparing: 100%|██████████| 3567/3567 [00:27<00:00, 128.49img/s]


  Near-duplicates removed  : 18
  Remaining images         : 3,549

  Final label distribution:
    pathology
    MALIGNANT                  1450
    BENIGN                     1437
    BENIGN_WITHOUT_CALLBACK     662

▶  STEP 5 — Patient-level stratified split …

  ┌─ Why patient-level? ──────────────────────────────────────────┐
  │  Each patient can have multiple cropped images (different     │
  │  views, different abnormalities). Splitting by IMAGE would    │
  │  put the same patient in both train and test → the model      │
  │  learns patient anatomy, not tumour features → data leakage. │
  │  Splitting by PATIENT guarantees zero overlap.                │
  └───────────────────────────────────────────────────────────────┘

  Labelled images   : 3,549
  Unlabelled images : 0

  Total unique patients : 1,566
  Malignant patients    : 748  (47.8%)
  Benign patients       : 818  (52.2%)

  ━  ━  ━  ━  ━  ━  ━  ━  ━  ━  ━  ━  ━  ━  ━  ━  ━  ━  ━  ━  ━  ━  ━  ━  ━  ━  ━  ━  ━  ━  ━  

  Copying: 100%|██████████| 3549/3549 [02:08<00:00, 27.53img/s]



  Copied : 3,549
  Errors : 0

▶  STEP 7 — Saving CSV manifests …
  ✔ train_manifest.csv  (2,476 rows)
  ✔ val_manifest.csv  (571 rows)
  ✔ test_manifest.csv  (502 rows)
  ✔ full_manifest.csv  (3,549 rows)

  FINAL AUDIT REPORT

  ┌─ Image pipeline ──────────────────────────────────────┐
  │  Cropped images in dicom_info     : 3,567           │
  │  Files found on Drive             : 3,567           │
  │  After exact-duplicate removal    : 3,567  (-0)     │
  │  After near-duplicate removal     : 3,549  (-18)     │
  │  Labelled images (used)           : 3,549           │
  └───────────────────────────────────────────────────────┘

  Split    Images  Malignant   Benign   Patients  Mal%
  ───────────────────────────────────────────────────────
  train     2,476      1,004    1,472      1,096  40.5%
  val         571        231      340        235  40.5%
  test        502        215      287        235  42.8%

  🔒 Leakage audit:
    Patient-level  Train∩Val=0  Train∩Test=0  Val∩Test=0


In [ ]:
# ============================================================
# DATA LEAKAGE AUDIT CELL
# ============================================================

import pandas as pd

print("="*60)
print("DATA LEAKAGE VERIFICATION")
print("="*60)

# ------------------------------------------------------------
# 1. LOAD SPLIT CSV FILES
# ------------------------------------------------------------

train_df = pd.read_csv(OUTPUT_DIR / "train_manifest.csv")
val_df   = pd.read_csv(OUTPUT_DIR / "val_manifest.csv")
test_df  = pd.read_csv(OUTPUT_DIR / "test_manifest.csv")

print("\nDataset sizes")
print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

# ------------------------------------------------------------
# 2. PATIENT LEAKAGE CHECK
# ------------------------------------------------------------

train_patients = set(train_df["patient_id"])
val_patients   = set(val_df["patient_id"])
test_patients  = set(test_df["patient_id"])

train_val_overlap  = train_patients & val_patients
train_test_overlap = train_patients & test_patients
val_test_overlap   = val_patients & test_patients

print("\nPATIENT LEVEL LEAKAGE CHECK")
print("Train ∩ Val  :", len(train_val_overlap))
print("Train ∩ Test :", len(train_test_overlap))
print("Val ∩ Test   :", len(val_test_overlap))

if len(train_val_overlap)==0 and len(train_test_overlap)==0 and len(val_test_overlap)==0:
    print("✅ No patient leakage detected")
else:
    print("❌ Patient leakage detected!")

# ------------------------------------------------------------
# 3. IMAGE PATH LEAKAGE CHECK
# ------------------------------------------------------------

train_images = set(train_df["new_filepath"])
val_images   = set(val_df["new_filepath"])
test_images  = set(test_df["new_filepath"])

print("\nIMAGE LEVEL LEAKAGE CHECK")

print("Train ∩ Val  :", len(train_images & val_images))
print("Train ∩ Test :", len(train_images & test_images))
print("Val ∩ Test   :", len(val_images & test_images))

if len(train_images & val_images)==0 and len(train_images & test_images)==0 and len(val_images & test_images)==0:
    print("✅ No image overlap detected")
else:
    print("❌ Image overlap detected")

# ------------------------------------------------------------
# 4. DUPLICATE HASH CHECK
# ------------------------------------------------------------

train_hash = set(train_df["md5"])
val_hash   = set(val_df["md5"])
test_hash  = set(test_df["md5"])

print("\nDUPLICATE IMAGE HASH CHECK")

print("Train ∩ Val  :", len(train_hash & val_hash))
print("Train ∩ Test :", len(train_hash & test_hash))
print("Val ∩ Test   :", len(val_hash & test_hash))

if len(train_hash & val_hash)==0 and len(train_hash & test_hash)==0 and len(val_hash & test_hash)==0:
    print("✅ No duplicate image leakage")
else:
    print("❌ Duplicate images exist across splits")

# ------------------------------------------------------------
# 5. CLASS DISTRIBUTION CHECK
# ------------------------------------------------------------

print("\nCLASS DISTRIBUTION")

print("\nTrain")
print(train_df["label"].value_counts())

print("\nVal")
print(val_df["label"].value_counts())

print("\nTest")
print(test_df["label"].value_counts())

# ------------------------------------------------------------
# FINAL RESULT
# ------------------------------------------------------------

print("\n"+"="*60)
print("FINAL RESULT")

if (len(train_val_overlap)==0 and
    len(train_test_overlap)==0 and
    len(val_test_overlap)==0 and
    len(train_images & val_images)==0 and
    len(train_images & test_images)==0 and
    len(val_images & test_images)==0):

    print("✅ DATASET IS SAFE — NO DATA LEAKAGE")

else:
    print("❌ DATA LEAKAGE DETECTED")

print("="*60)

DATA LEAKAGE VERIFICATION

Dataset sizes
Train: 2476
Val: 571
Test: 502

PATIENT LEVEL LEAKAGE CHECK
Train ∩ Val  : 0
Train ∩ Test : 0
Val ∩ Test   : 0
✅ No patient leakage detected

IMAGE LEVEL LEAKAGE CHECK
Train ∩ Val  : 0
Train ∩ Test : 0
Val ∩ Test   : 0
✅ No image overlap detected

DUPLICATE IMAGE HASH CHECK
Train ∩ Val  : 0
Train ∩ Test : 0
Val ∩ Test   : 0
✅ No duplicate image leakage

CLASS DISTRIBUTION

Train
label
0    1472
1    1004
Name: count, dtype: int64

Val
label
0    340
1    231
Name: count, dtype: int64

Test
label
0    287
1    215
Name: count, dtype: int64

FINAL RESULT
✅ DATASET IS SAFE — NO DATA LEAKAGE
